# 개별종목 조합J — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합J 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합J의 피처 값만 지정합니다.
import json

COMBINATION = 'J'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
    'relative_ret_5_market',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합J 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return', 'relative_ret_5_market')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.5023,0.5012,0.0010,0.2808,0.3506,0.0670,0.3805,0.0425,0.1031
1,2,balanced,980,20150123,20150421,0.4032,0.3978,0.0054,0.3535,0.3708,0.0702,0.3862,0.1884,0.2826
2,3,balanced,1210,20151228,20160328,0.3714,0.3762,-0.0048,0.3680,0.3677,0.0546,0.3774,0.3267,0.3542
3,4,balanced,1439,20161202,20170228,0.4673,0.4617,0.0056,0.3546,0.3812,0.1024,0.4123,0.1495,0.2576
4,5,balanced,1669,20171113,20180207,0.4287,0.3901,0.0386,0.3905,0.4036,0.1190,0.4062,0.2725,0.3504
5,6,balanced,1899,20181024,20190118,0.4135,0.3725,0.0411,0.4133,0.4186,0.1308,0.4192,0.4654,0.4294
6,7,balanced,2129,20190930,20191224,0.4713,0.4781,-0.0068,0.3359,0.3701,0.0856,0.4141,0.1662,0.2699
7,8,balanced,2359,20200902,20201130,0.4129,0.3476,0.0653,0.4118,0.4144,0.1217,0.4108,0.4295,0.4179
8,9,balanced,2589,20210806,20211105,0.4027,0.3914,0.0113,0.3876,0.3944,0.0888,0.3860,0.3149,0.3641
9,10,balanced,2818,20220714,20221012,0.3527,0.3454,0.0072,0.3494,0.3543,0.0333,0.3737,0.2676,0.3180


,OOS 폴드 평균
accuracy,0.4197
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0228
macro_f1,0.3700
balanced_accuracy,0.3855
mcc,0.0902
pr_auc_macro_ovr,0.3976
down_recall,0.2709
core_harmonic_mean,0.3234


재실행 명령: python scripts/run_stock_model_experiment.py
